# Filter Concepts

This notebook spins the voice-band filtering material out of `am_vs_fm_audio_demo.ipynb` into a standalone lesson on low-pass, high-pass, and band-pass behavior.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
VOICE_FILE = ROOT / "assets" / "local" / "my_voice.m4a"
WORK_FS = 96_000
PLAY_FS = 44_100

if VOICE_FILE.exists():
    raw_fs, raw_audio = load_audio(VOICE_FILE, normalize_audio=True)
    raw_audio = normalize(ensure_mono(raw_audio))
    raw_audio = raw_audio[: int(raw_fs * 5)]
    voice_work = normalize(resample_signal(raw_audio, raw_fs, WORK_FS))
    voice_play = normalize(resample_signal(raw_audio, raw_fs, PLAY_FS))
    print(f"Loaded {VOICE_FILE.name} at {raw_fs} Hz")
else:
    t_fallback = np.arange(0, 3.0, 1 / WORK_FS)
    voice_work = normalize(
        0.7 * np.sin(2 * np.pi * 220 * t_fallback)
        + 0.4 * np.sin(2 * np.pi * 440 * t_fallback)
        + 0.2 * np.sin(2 * np.pi * 880 * t_fallback)
    )
    voice_play = normalize(resample_signal(voice_work, WORK_FS, PLAY_FS))
    print(f"No local voice recording found at {VOICE_FILE}. Using a synthetic fallback.")

t_work = np.arange(len(voice_work)) / WORK_FS


## Why Radios Filter Audio

Most narrowband voice links only need about 300 Hz to 3 kHz of audio. Everything below that wastes deviation and everything above that mostly carries hiss or sharp transients.

In [ ]:
voice_bp = signal.sosfilt(
    signal.butter(5, [300, 3000], btype="band", fs=WORK_FS, output="sos"),
    voice_work,
)
voice_bp = normalize(voice_bp)

fig, axes = plt.subplots(2, 2, figsize=(13, 6))
plot_waveform(voice_work[:10_000], fs=WORK_FS, ax=axes[0, 0], title="Original Voice")
plot_waveform(voice_bp[:10_000], fs=WORK_FS, ax=axes[0, 1], title="Band-Limited Voice")
plot_spectrum(voice_work, fs=WORK_FS, ax=axes[1, 0], title="Original Spectrum")
plot_spectrum(voice_bp, fs=WORK_FS, ax=axes[1, 1], title="Band-Limited Spectrum")
axes[1, 0].set_xlim(0, 8000)
axes[1, 1].set_xlim(0, 8000)
axes[1, 0].set_ylim(-100, 5)
axes[1, 1].set_ylim(-100, 5)
plt.tight_layout()

display(Markdown("**Original audio**"))
display(audio_player(resample_signal(voice_work, WORK_FS, PLAY_FS), rate=PLAY_FS))
display(Markdown("**Band-limited audio**"))
display(audio_player(resample_signal(voice_bp, WORK_FS, PLAY_FS), rate=PLAY_FS))


## Interactive Low-Pass Filtering

Dragging the cutoff down is the quickest way to hear what filters do. As the cutoff falls, intelligibility gives way to muffled energy and then mostly pitch contour.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_filter(cutoff=2500.0):
    sos = signal.butter(5, cutoff, btype="low", fs=WORK_FS, output="sos")
    filtered = normalize(signal.sosfilt(sos, voice_work))
    axes[0].clear()
    axes[1].clear()
    plot_waveform(filtered[:10_000], fs=WORK_FS, ax=axes[0], title=f"Low-pass, cutoff={cutoff:.0f} Hz")
    plot_spectrum(filtered, fs=WORK_FS, ax=axes[1], title="Filtered Spectrum")
    axes[1].set_xlim(0, 8000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(filtered, WORK_FS, PLAY_FS), rate=PLAY_FS)

controls = widgets.interactive(
    update_filter,
    cutoff=float_slider(min_value=300, max_value=5000, step=100, value=2500, description="Cutoff"),
)
display(controls, audio_out)


## Key Takeaway

Filters are deliberate frequency selectors. The radio "sound" you hear is not just the microphone or speaker; it is shaped by bandwidth limits all along the chain.